<a href="https://colab.research.google.com/github/AsimHanif150/Ai-ML_internship_01-/blob/main/weeks-3-full-release-duckdb-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip -q install duckdb huggingface_hub

In [5]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [ ]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients':
        f"read_parquet('{REL}/dim_clients.parquet')",

    'dim_content':
        f"read_parquet('{REL}/dim_content.parquet')",

    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_daily_sample':
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [ ]:
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_clients.parquet' (HTTP 401)

In [ ]:
import getpass
import duckdb

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face token configured successfully.")

Paste your Hugging Face READ token (hf_...): ··········
Hugging Face token configured successfully.


In [ ]:
test = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
    )
    LIMIT 5
""").df()

test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [ ]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print(f"{len(features_90d):,} content items with enough history")

features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,2298.0,0.0,11.779379
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,13712.0,44.0,6.959208
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,251.0,0.0,48.972195
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,21066.0,12.0,28.140689
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,1919.0,5.0,20.731918


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)



clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.552     0.341     0.421      9389
           1      0.687     0.839     0.755     16162

    accuracy                          0.656     25551
   macro avg      0.619     0.590     0.588     25551
weighted avg      0.637     0.656     0.633     25551



In [ ]:
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

GatedRepoError: 401 Client Error. (Request ID: Root=1-6aad2d56-6002e76441631ba87878b3b9;5933b3ee-4e85-43ec-8585-f82d04313330)

Cannot access gated repo for url https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet.
Access to dataset FlyRank/internship-warehouse is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "dim_*.parquet",
        "fact_content_query_90d.parquet",
        "fact_content_daily_performance/month=2026-0*/*.parquet"
    ]
)

print(path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [ ]:
REL = path

TABLES = {
    'dim_clients':
        f"read_parquet('{REL}/dim_clients.parquet')",

    'dim_content':
        f"read_parquet('{REL}/dim_content.parquet')",

    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [ ]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

dim_clients                   104 rows
dim_content               519,606 rows
fact_daily             58,893,481 rows
fact_query_90d          2,414,248 rows


In [6]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [7]:
REL = 'hf://datasets/FlyRank/internship-warehouse'

TABLES = {
    'dim_clients':
        f"read_parquet('{REL}/dim_clients.parquet')",

    'dim_content':
        f"read_parquet('{REL}/dim_content.parquet')",

    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_daily_sample':
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [8]:
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

HTTPException: HTTP Error: HTTP GET error on 'https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/dim_clients.parquet' (HTTP 401)

In [9]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

DatasetNotFoundError: Dataset 'FlyRank/internship-warehouse' is a gated dataset on the Hub. You must be authenticated to access it.

In [12]:
import getpass
import duckdb

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face token configured successfully.")

Paste your Hugging Face READ token (hf_...): ··········
Hugging Face token configured successfully.


In [13]:
test = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet'
    )
    LIMIT 5
""").df()

test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [14]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 60 DAY
                     AND f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']} f, bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev30 >= 100
    )

    SELECT *
    FROM windowed
""").df()

print(f"{len(features_90d):,} content items with enough history")

features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,2298.0,0.0,11.779379
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,13712.0,44.0,6.959208
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,251.0,0.0,48.972195
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,21066.0,12.0,28.140689
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,1919.0,5.0,20.731918


In [16]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)



clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [17]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


In [18]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


In [19]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.552     0.341     0.421      9389
           1      0.687     0.839     0.755     16162

    accuracy                          0.656     25551
   macro avg      0.619     0.590     0.588     25551
weighted avg      0.637     0.656     0.633     25551



In [20]:
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

GatedRepoError: 401 Client Error. (Request ID: Root=1-6aad2d56-6002e76441631ba87878b3b9;5933b3ee-4e85-43ec-8585-f82d04313330)

Cannot access gated repo for url https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/dim_clients.parquet.
Access to dataset FlyRank/internship-warehouse is restricted. You must have access to it and be authenticated to access it. Please log in.

In [21]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "dim_*.parquet",
        "fact_content_query_90d.parquet",
        "fact_content_daily_performance/month=2026-0*/*.parquet"
    ]
)

print(path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [22]:
REL = path

TABLES = {
    'dim_clients':
        f"read_parquet('{REL}/dim_clients.parquet')",

    'dim_content':
        f"read_parquet('{REL}/dim_content.parquet')",

    'fact_daily':
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    'fact_query_90d':
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [23]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:20} {n:>12,} rows")

dim_clients                   104 rows
dim_content               519,606 rows
fact_daily             58,893,481 rows
fact_query_90d          2,414,248 rows
